# RAG Pipeline: Psychoeducation Knowledge Base

Corpus files under `data/rag_corpus/` (see `src/rag_pipeline.load_corpus`).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

corpus_dir = ROOT / "data" / "rag_corpus"
rows = []
for p in sorted(corpus_dir.rglob("*")):
    if p.is_file() and p.suffix.lower() in {".txt", ".md"}:
        text = p.read_text(encoding="utf-8", errors="replace")
        wc = len(text.split())
        rows.append((p.name, wc))
for name, wc in rows:
    print(f"{name}\twords={wc}")
print(f"\nTotal files: {len(rows)}")


# Vector Store Creation

Chunks documents and builds or loads **ChromaDB** (`src/rag_pipeline.chunk_documents`, `create_vectorstore`, `load_vectorstore`) — same flow as `src/app.py`.

In [ ]:
import yaml
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.rag_pipeline import chunk_documents, create_vectorstore, load_corpus, load_vectorstore

cfg_path = ROOT / "config" / "config.yaml"
with open(cfg_path, encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}
rag_cfg = cfg.get("rag") or {}
embed_model = rag_cfg.get("embedding_model", "sentence-transformers/all-MiniLM-L6-v2")
persist = ROOT / "chroma_db"
chunk_size = int(rag_cfg.get("chunk_size", 500))
overlap = int(rag_cfg.get("chunk_overlap", 50))

docs = load_corpus(ROOT / "data" / "rag_corpus")
chunks = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=overlap)
print(f"Documents loaded: {len(docs)}")
print(f"Chunks (split config): {len(chunks)}")

if persist.exists() and any(persist.iterdir()):
    vs = load_vectorstore(embedding_model=embed_model, persist_directory=persist)
    print("Using persisted vectorstore at", persist)
else:
    vs = create_vectorstore(chunks, embedding_model=embed_model, persist_directory=persist)
    print("Created vectorstore at", persist)

print("Expected (paper): 17 docs, 341 chunks")


# Retrieval Test

`retrieve()` — same as `src/rag_pipeline.retrieve`.

In [ ]:
from src.rag_pipeline import retrieve

queries = [
    "What are symptoms of depression?",
    "How can I cope with anxiety?",
    "What is cognitive behavioral therapy?",
    "Signs of burnout and stress",
    "Sleep problems and mental health",
]
for q in queries:
    print("=" * 60)
    print("QUERY:", q)
    hits = retrieve(vs, q, k=3)
    for i, doc in enumerate(hits, 1):
        title = doc.metadata.get("title", "untitled")
        preview = (doc.page_content or "")[:350].replace("\n", " ")
        print(f"  [{i}] {title}: {preview}...")
